# GEDI Footprint Analysis Workflow (2 of 2 — Analysis)

Picks up where `gedi_footprint_1_extract.ipynb` left off. Instead of re-running the GEDI search/filter/clip steps, this notebook loads the **clipped points and footprints GeoJSON files** that notebook 1 saved.

Run the two cells below first (imports, then the file picker), then continue through the Monte Carlo simulation and CHM comparison sections in order.

In [1]:
#------------ imports ----------------------------------------------------------
import os
import math
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, mapping
from pyproj import Transformer
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog

import rasterio
from rasterio.windows import from_bounds as window_from_bounds
from rasterio.mask import mask as rio_mask
from rasterio.plot import plotting_extent
from shapely.geometry import mapping

---
## 2 - Load Saved GEDI Data

Select the **clipped points GeoJSON** file that notebook 1 saved (named like `<header>_gedi_shots_clipped.geojson`). A file-picker dialog will open. This notebook will then automatically look for the matching clipped footprints file (`<header>_gedi_footprints_clipped.geojson`) in the same folder — if it can't find it, you'll be asked to pick that one too.

In [2]:
# ── Select and load the saved, clipped GEDI outputs from notebook 1 ────────────
def select_file(title, filetypes):
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    path = filedialog.askopenfilename(title=title, filetypes=filetypes)
    root.destroy()
    if not path:
        raise SystemExit(f'No file selected for: {title}')
    return path

print('Select the CLIPPED POINTS GeoJSON file saved by notebook 1')
print("(named like '<header>_gedi_shots_clipped.geojson')")
clipped_points_path = select_file(
    'Select clipped GEDI points GeoJSON',
    [('GeoJSON files', '*.geojson'), ('All files', '*.*')]
)

data_folder = os.path.dirname(clipped_points_path)
fname = os.path.basename(clipped_points_path)
file_header = fname.replace('_gedi_shots_clipped.geojson', '')

# Try to auto-locate the matching clipped footprints file in the same folder
guess_footprints_path = os.path.join(data_folder, f'{file_header}_gedi_footprints_clipped.geojson')

if os.path.exists(guess_footprints_path):
    clipped_footprints_path = guess_footprints_path
    print(f'\nAuto-found matching footprints file: {clipped_footprints_path}')
else:
    print('\nCould not auto-locate the matching footprints file — please select it.')
    clipped_footprints_path = select_file(
        'Select clipped GEDI footprints GeoJSON',
        [('GeoJSON files', '*.geojson'), ('All files', '*.*')]
    )


gdf_points_clipped = gpd.read_file(clipped_points_path)
gdf_footprints_clipped = gpd.read_file(clipped_footprints_path)

# Ensure shot_number stays as string (ArcGIS Pro compatibility, and needed
# for merges further down)
gdf_points_clipped['shot_number'] = gdf_points_clipped['shot_number'].astype(str)
gdf_footprints_clipped['shot_number'] = gdf_footprints_clipped['shot_number'].astype(str)

# Drop any leftover reading_order_id from a previous run so Step A below
# regenerates it cleanly
gdf_points_clipped = gdf_points_clipped.drop(columns='reading_order_id', errors='ignore')
gdf_footprints_clipped = gdf_footprints_clipped.drop(columns='reading_order_id', errors='ignore')

WGS84 = 'EPSG:4326'

print(f'\nLoaded:')
print(f'  data_folder = {data_folder}')
print(f'  file_header = {file_header}')
print(f'  Points     : {len(gdf_points_clipped):,} rows from {clipped_points_path}')
print(f'  Footprints : {len(gdf_footprints_clipped):,} rows from {clipped_footprints_path}')

Select the CLIPPED POINTS GeoJSON file saved by notebook 1
(named like '<header>_gedi_shots_clipped.geojson')

Auto-found matching footprints file: C:/Users/davisk10/OneDrive - Cal Poly/Tree Biomass Estimation Research - Documents/CODE/Notebook_Test1_Arb_Output_Files\arb_test_1_gedi_footprints_clipped.geojson

Loaded:
  data_folder = C:/Users/davisk10/OneDrive - Cal Poly/Tree Biomass Estimation Research - Documents/CODE/Notebook_Test1_Arb_Output_Files
  file_header = arb_test_1
  Points     : 6 rows from C:/Users/davisk10/OneDrive - Cal Poly/Tree Biomass Estimation Research - Documents/CODE/Notebook_Test1_Arb_Output_Files/arb_test_1_gedi_shots_clipped.geojson
  Footprints : 6 rows from C:/Users/davisk10/OneDrive - Cal Poly/Tree Biomass Estimation Research - Documents/CODE/Notebook_Test1_Arb_Output_Files\arb_test_1_gedi_footprints_clipped.geojson


## DONT RUN FOR NOW (Messing Around with Graph Ouputs for GEDI Data)

In [ ]:
# # ── RH Canopy Profile — One Chart Per Footprint ───────────────────────────────
# # Horizontal bar chart showing each rh metric as a canopy height for that footprint.
# # One figure saved per footprint. UAV CHM overlay to be added later.

# import matplotlib.pyplot as plt
# import math

# rh_cols        = ['rh25', 'rh50', 'rh75', 'rh95', 'rh98', 'rh99', 'rh100']
# rh_percentiles = [25,      50,     75,     95,     98,     99,     100]

# for plot_idx, (_, row) in enumerate(gdf_points_clipped.iterrows()):

#     fig, ax = plt.subplots(figsize=(6, 7))

#     rh_values = [row[col] for col in rh_cols]

#     # Height intervals between consecutive rh percentiles
#     heights   = [0] + rh_values
#     intervals = [heights[i+1] - heights[i] for i in range(len(rh_values))]
#     percentile_widths = [rh_percentiles[0]] + [
#         rh_percentiles[i] - rh_percentiles[i-1] for i in range(1, len(rh_percentiles))
#     ]

#     ax.barh(
#         y      = [heights[i] + intervals[i] / 2 for i in range(len(intervals))],
#         width  = percentile_widths,
#         height = intervals,
#         color='steelblue', edgecolor='white', linewidth=0.5, alpha=0.75,
#         label='GEDI RH profile'
#     )

#     # Label each rh metric
#     for rh_val, rh_pct in zip(rh_values, rh_percentiles):
#         ax.axhline(rh_val, color='black', linewidth=0.6, linestyle='--', alpha=0.4)
#         ax.text(max(percentile_widths) * 0.98, rh_val + 0.2,
#                 f'rh{rh_pct} = {rh_val:.1f} m',
#                 fontsize=7, ha='right', va='bottom')

#     ax.set_xlabel('Percentile interval width (%)', fontsize=9)
#     ax.set_ylabel('Height (m)', fontsize=9)
#     ax.set_ylim(0, max(rh_values) * 1.15)
#     ax.set_title(
#         f'Footprint {plot_idx + 1} — Canopy Height Profile\n'
#         f'Date: {row["date"]}  |  Beam: {row["beam"]}\n'
#         f'rh98: {row["rh98"]:.1f} m  |  Sensitivity: {row["sensitivity"]:.2f}',
#         fontsize=9
#     )
#     ax.legend(fontsize=8)
#     ax.grid(axis='x', linestyle='--', alpha=0.3)
#     ax.tick_params(labelsize=8)

#     plt.tight_layout()

#     #optional for saving the graphs created
#     #save_path = os.path.join(data_folder, f'{file_header}_footprint{plot_idx + 1}_rh_profile.png')
#     #plt.savefig(save_path, dpi=150, bbox_inches='tight')
#     #print(f"Saved: {save_path}")

#     plt.show()

#     # ── Raw GEDI Waveform Plot — One Chart Per Footprint ─────────────────────────
# # Plots the raw received waveform (rxwaveform) for each clipped footprint.
# # The waveform shows laser return intensity vs sample index (proxy for height) —
# # peaks indicate surfaces where the laser reflected (ground, understory, canopy).
# # This is the black line from the reference figure.

# for plot_idx, (_, row) in enumerate(gdf_points_clipped.iterrows()):

#     sn = row['shot_number']

#     if sn not in waveform_store:
#         print(f"Footprint {plot_idx + 1}: no waveform found for shot {sn}")
#         continue

#     waveform = waveform_store[sn]

#     # Sample index runs top-down (first sample = top of atmosphere,
#     # last sample = ground), so we flip it so ground is at the bottom
#     samples = np.arange(len(waveform))
#     samples_flipped = samples[::-1]

#     fig, ax = plt.subplots(figsize=(5, 7))

#     ax.plot(
#         waveform, samples_flipped,
#         color='black', linewidth=1.2, label='GEDI waveform'
#     )

#     # Shade the area under the waveform like the reference figure
#     ax.fill_betweenx(
#         samples_flipped, 0, waveform,
#         color='steelblue', alpha=0.2
#     )

#     ax.set_xlabel('Return intensity (counts)', fontsize=9)
#     ax.set_ylabel('Sample index (↑ = higher in canopy)', fontsize=9)
#     ax.set_title(
#         f'Footprint {plot_idx + 1} — Raw GEDI Waveform\n'
#         f'Date: {row["date"]}  |  Beam: {row["beam"]}\n'
#         f'rh98: {row["rh98"]:.1f} m  |  Sensitivity: {row["sensitivity"]:.2f}',
#         fontsize=9
#     )
#     ax.legend(fontsize=8)
#     ax.grid(axis='x', linestyle='--', alpha=0.3)
#     ax.tick_params(labelsize=8)

#     plt.tight_layout()
#     plt.show()

## Assign Labels for GEDI Footprints

In [3]:
# ════════════════════════════════════════════════════════════════════════════
# Assign reading-order numbers (top-to-bottom, left-to-right)
# ════════════════════════════════════════════════════════════════════════════
# Groups points into "rows" based on latitude proximity, then numbers each
# row left-to-right (low lon → high lon), rows ordered top-to-bottom (high lat → low lat)

ROW_TOLERANCE_M = 15.0   # points within this many metres of latitude count as the same "row"
                          # tune this based on your point spacing — too small splits a row
                          # into multiple rows, too large merges separate rows together

row_tol_deg = ROW_TOLERANCE_M / 111320.0   # ~111,320 m per degree latitude

# Ensure shot_number is a consistent string type before any merging
gdf_points_clipped['shot_number'] = gdf_points_clipped['shot_number'].astype(str)

gdf_sorted = gdf_points_clipped.sort_values('lat', ascending=False).reset_index(drop=True)

row_ids = []
current_row = 0
row_ref_lat = gdf_sorted.loc[0, 'lat']

for lat_val in gdf_sorted['lat']:
    if row_ref_lat - lat_val > row_tol_deg:
        current_row += 1
        row_ref_lat = lat_val
    row_ids.append(current_row)

gdf_sorted['row_id'] = row_ids
gdf_sorted = gdf_sorted.sort_values(['row_id', 'lon'], ascending=[True, True]).reset_index(drop=True)
gdf_sorted['reading_order_id'] = gdf_sorted.index + 1   # start at 1, not 0

gdf_points_clipped = gdf_points_clipped.merge(
    gdf_sorted[['shot_number', 'reading_order_id']],
    on='shot_number',
    how='left'
)

n_missing_a = gdf_points_clipped['reading_order_id'].isna().sum()
if n_missing_a > 0:
    print(f"WARNING: {n_missing_a} points in gdf_points_clipped have no reading_order_id — check shot_number dtypes")

print(gdf_points_clipped[['shot_number', 'lat', 'lon', 'reading_order_id']]
      .sort_values('reading_order_id').to_string(index=False))
# ════════════════════════════════════════════════════════════════════════════
# Propagate reading_order_id onto footprints, re-save GeoJSON files
# ════════════════════════════════════════════════════════════════════════════
# gdf_footprints_clipped doesn't have reading_order_id yet — it was only
# assigned to gdf_points_clipped in Step A. Merge it across using shot_number,
# then overwrite both clipped GeoJSON files so the attribute table includes
# the new column when opened in QGIS/ArcGIS.

gdf_footprints_clipped['shot_number'] = gdf_footprints_clipped['shot_number'].astype(str)

if 'reading_order_id' in gdf_footprints_clipped.columns:
    gdf_footprints_clipped = gdf_footprints_clipped.drop(columns='reading_order_id')

reading_id_lookup = gdf_points_clipped[['shot_number', 'reading_order_id']].drop_duplicates()

gdf_footprints_clipped = gdf_footprints_clipped.merge(
    reading_id_lookup, on='shot_number', how='left'
)

n_missing_fp = gdf_footprints_clipped['reading_order_id'].isna().sum()
if n_missing_fp > 0:
    print(f"WARNING: {n_missing_fp} footprints did not get a reading_order_id")
else:
    print("reading_order_id successfully added to gdf_footprints_clipped")

# Re-save both clipped GeoJSON files, now including reading_order_id
gdf_footprints_clipped.to_file(clipped_footprints_path, driver='GeoJSON')
gdf_points_clipped.to_file(clipped_points_path, driver='GeoJSON')

print(f"Footprints GeoJSON updated: {clipped_footprints_path}")
print(f"Points GeoJSON updated:     {clipped_points_path}")

       shot_number       lat         lon  reading_order_id
151690300300267578 35.310809 -120.662771                 1
 76920300200076713 35.310589 -120.662524                 2
313210300200185367 35.310438 -120.662134                 3
357460600200326393 35.310392 -120.661510                 4
 76920300200076712 35.310242 -120.662984                 5
184280800200188204 35.310237 -120.661568                 6
reading_order_id successfully added to gdf_footprints_clipped
Footprints GeoJSON updated: C:/Users/davisk10/OneDrive - Cal Poly/Tree Biomass Estimation Research - Documents/CODE/Notebook_Test1_Arb_Output_Files\arb_test_1_gedi_footprints_clipped.geojson
Points GeoJSON updated:     C:/Users/davisk10/OneDrive - Cal Poly/Tree Biomass Estimation Research - Documents/CODE/Notebook_Test1_Arb_Output_Files/arb_test_1_gedi_shots_clipped.geojson


## Insert Manual Data & Pull CHM Percentiles 

Enter the CHM file genertaed from the R code here

Input Hypsometer and App Height Data here

Creates function for pulling percentile values from CHM 

In [4]:
# ── Helper: extract UAV CHM percentiles at any set of lon/lat points ────────
# ── Prompt user for the CHM GeoTIFF path ──────────────────────────────────────
chm_path = input("Enter the full path to the UAV CHM GeoTIFF file: ").strip().strip('"')  # remove quotes if user pasted a path with them

if not os.path.exists(chm_path):
    raise FileNotFoundError(f"CHM file not found at: {chm_path}")

def extract_uav_chm_percentiles(chm_path, lons, lats, percentiles, buffer_radius_m=12.5, progress_every=500):
    """For each (lon, lat) pair (WGS84 degrees), extract a circular buffer
    from the UAV CHM GeoTIFF and compute each requested percentile of CHM
    height within that buffer. Returns a dict of {percentile: np.ndarray}."""
    n_total = len(lons)
    results = {p: np.full(n_total, np.nan) for p in percentiles}
    n_outside = 0
    n_nodata = 0

    with rasterio.open(chm_path) as chm_src:
        chm_nodata = chm_src.nodata
        to_chm_crs = Transformer.from_crs(WGS84, chm_src.crs, always_xy=True)

        for i, (lon, lat) in enumerate(zip(lons, lats)):
            x_chm, y_chm = to_chm_crs.transform(lon, lat)
            buffer_geom = Point(x_chm, y_chm).buffer(buffer_radius_m)

            try:
                out_image, _ = rio_mask(
                    chm_src, [mapping(buffer_geom)], crop=True, filled=True, nodata=chm_nodata
                )
            except ValueError:
                n_outside += 1
                continue

            chm_values = out_image[0]
            valid_values = chm_values[chm_values != chm_nodata] if chm_nodata is not None else chm_values.flatten()
            valid_values = valid_values[~np.isnan(valid_values)]

            if valid_values.size == 0:
                n_nodata += 1
                continue

            for p in percentiles:
                results[p][i] = np.percentile(valid_values, p)

            if progress_every and (i + 1) % progress_every == 0:
                print(f"  Processed {i + 1:,} / {n_total:,} positions...")

    print(f"CHM extraction complete — total: {n_total:,}, "
          f"outside CHM extent: {n_outside:,}, no valid data: {n_nodata:,}")
    return results


# ── Manually measured ground-truth heights (arboretum) ──────────────────────
# Keyed by reading_order_id (the Label numbers from the labeling cell above).
manual_ground_truth = {
    1: {'app_height_m': 14.37, 'hypsometer_height_m': 13.9},
    2: {'app_height_m': 24.81, 'hypsometer_height_m': 23.7},
    3: {'app_height_m': 15.74, 'hypsometer_height_m': 15.2},
    4: {'app_height_m': 14.39, 'hypsometer_height_m': 14.1},
    5: {'app_height_m': 18.22, 'hypsometer_height_m': 17.9},
    6: {'app_height_m': 10.51, 'hypsometer_height_m': 10.0},
}

gdf_ground_truth = (
    pd.DataFrame.from_dict(manual_ground_truth, orient='index')
    .rename_axis('reading_order_id')
    .reset_index()
)

print(f"Manually measured ground-truth heights loaded for {len(gdf_ground_truth)} footprints:")
display(gdf_ground_truth)

Manually measured ground-truth heights loaded for 6 footprints:


,reading_order_id,app_height_m,hypsometer_height_m
0,1,14.37,13.9
1,2,24.81,23.7
2,3,15.74,15.2
3,4,14.39,14.1
4,5,18.22,17.9
5,6,10.51,10.0


# ── Stage 1: which UAV percentile (P95-P100) is closest to hypsometer? ──────
UAV_PERCENTILES = list(range(95, 101))

uav_95_100 = extract_uav_chm_percentiles(
    chm_path, gdf_footprints_clipped['lon'], gdf_footprints_clipped['lat'],
    percentiles=UAV_PERCENTILES, buffer_radius_m=BUFFER_RADIUS_M, progress_every=None
)
for p in UAV_PERCENTILES:
    gdf_footprints_clipped[f'p{p}_uav_chm'] = uav_95_100[p]

uav_vs_hyps = gdf_footprints_clipped[['reading_order_id'] + [f'p{p}_uav_chm' for p in UAV_PERCENTILES]].merge(
    gdf_ground_truth[['reading_order_id', 'hypsometer_height_m']], on='reading_order_id', how='inner'
)

uav_ranking = pd.DataFrame([
    {
        'UAV_Percentile': p,
        'MAE_m': (uav_vs_hyps[f'p{p}_uav_chm'] - uav_vs_hyps['hypsometer_height_m']).abs().mean(),
        'RMSE_m': np.sqrt(((uav_vs_hyps[f'p{p}_uav_chm'] - uav_vs_hyps['hypsometer_height_m']) ** 2).mean()),
    }
    for p in UAV_PERCENTILES
]).sort_values('MAE_m').reset_index(drop=True)

best_uav_percentile = int(uav_ranking.iloc[0]['UAV_Percentile'])
print(f"Best UAV percentile vs. hypsometer: P{best_uav_percentile}  "
      f"(MAE={uav_ranking.iloc[0]['MAE_m']:.2f} m, RMSE={uav_ranking.iloc[0]['RMSE_m']:.2f} m)")
display(uav_ranking)

In [5]:

# ── Stage 1: which UAV percentile (P95-P100) is closest to hypsometer? ──────
#first build the helper table
def build_percentile_comparison_table(df, percentiles, height_col_template, height_col_label,
                                       ground_truth_col='hypsometer_height_m',
                                       ground_truth_label='Hyps H (m)',
                                       id_col='reading_order_id'):
    """
    Side-by-side, per-footprint table: rows = percentiles (100 down to the
    lowest requested), columns = (Footprint N, <ground_truth_label>) and
    (Footprint N, <height_col_label>) for every footprint in df.
    """
    percentiles_sorted = sorted(percentiles, reverse=True)
    footprint_ids = sorted(df[id_col].dropna().unique())

    table_data = {}
    for fid in footprint_ids:
        row = df[df[id_col] == fid].iloc[0]
        label = f'Footprint {int(fid)}'
        table_data[(label, ground_truth_label)] = [row[ground_truth_col]] * len(percentiles_sorted)
        table_data[(label, height_col_label)] = [
            row[height_col_template.format(p=p)] if height_col_template.format(p=p) in row else np.nan
            for p in percentiles_sorted
        ]

    table = pd.DataFrame(table_data, index=percentiles_sorted)
    table.index.name = 'Percentile'
    table.columns = pd.MultiIndex.from_tuples(table.columns)
    return table
# ── Stage 1: which UAV percentile (P95-P100) is closest to hypsometer? ──────
#build the ranking table based on the lowest MAE of each percentile vs hypsometer (RMSE included as well, not used for ranking)
UAV_PERCENTILES = list(range(95, 101))
BUFFER_RADIUS_M = 12.5   # 25 m GEDI footprint diameter → 12.5 m radius buffer


uav_95_100 = extract_uav_chm_percentiles(
    chm_path, gdf_footprints_clipped['lon'], gdf_footprints_clipped['lat'],
    percentiles=UAV_PERCENTILES, buffer_radius_m=BUFFER_RADIUS_M, progress_every=None
)
for p in UAV_PERCENTILES:
    gdf_footprints_clipped[f'p{p}_uav_chm'] = uav_95_100[p]

uav_vs_hyps = gdf_footprints_clipped[['reading_order_id'] + [f'p{p}_uav_chm' for p in UAV_PERCENTILES]].merge(
    gdf_ground_truth[['reading_order_id', 'hypsometer_height_m']], on='reading_order_id', how='inner'
)

uav_ranking = pd.DataFrame([
    {
        'UAV_Percentile': p,
        'MAE_m': (uav_vs_hyps[f'p{p}_uav_chm'] - uav_vs_hyps['hypsometer_height_m']).abs().mean(),
        'RMSE_m': np.sqrt(((uav_vs_hyps[f'p{p}_uav_chm'] - uav_vs_hyps['hypsometer_height_m']) ** 2).mean()),
    }
    for p in UAV_PERCENTILES
]).sort_values('MAE_m').reset_index(drop=True)

best_uav_percentile = int(uav_ranking.iloc[0]['UAV_Percentile'])
print(f"Best UAV percentile vs. hypsometer: P{best_uav_percentile}  "
      f"(MAE={uav_ranking.iloc[0]['MAE_m']:.2f} m, RMSE={uav_ranking.iloc[0]['RMSE_m']:.2f} m)")
display(uav_ranking)

# ── Stage 1 (App Height): rank UAV percentiles against App Height ───────────
uav_vs_app = (
    gdf_footprints_clipped[['reading_order_id'] + [f'p{p}_uav_chm' for p in UAV_PERCENTILES]]
    .merge(gdf_ground_truth[['reading_order_id', 'app_height_m']], on='reading_order_id', how='inner')
)


uav_ranking_app = pd.DataFrame([
    {
        'UAV_Percentile': p,
        'MAE_m': (uav_vs_app[f'p{p}_uav_chm'] - uav_vs_app['app_height_m']).abs().mean(),
        'MedAE_m': (uav_vs_app[f'p{p}_uav_chm'] - uav_vs_app['app_height_m']).abs().median(),
        'RMSE_m': np.sqrt(((uav_vs_app[f'p{p}_uav_chm'] - uav_vs_app['app_height_m']) ** 2).mean()),
    }
    for p in UAV_PERCENTILES
]).sort_values('MAE_m').reset_index(drop=True)

best_uav_percentile_app = int(uav_ranking_app.iloc[0]['UAV_Percentile'])
print(f"Best UAV percentile vs. App Height: P{best_uav_percentile_app}  "
      f"(MAE={uav_ranking_app.iloc[0]['MAE_m']:.2f} m, "
      f"MedAE={uav_ranking_app.iloc[0]['MedAE_m']:.2f} m, "
      f"RMSE={uav_ranking_app.iloc[0]['RMSE_m']:.2f} m)")
display(uav_ranking_app)


# ── Stage 1 table: percentile x footprint, hypsometer vs. UAV height ────────
stage1_table = build_percentile_comparison_table(
    uav_vs_hyps, UAV_PERCENTILES, 'p{p}_uav_chm', 'UAV H (m)'
)
display(stage1_table.round(2))

# Side-by-side per-footprint table, App Height vs. UAV height by percentile
stage1_app_table = build_percentile_comparison_table(
    uav_vs_app, UAV_PERCENTILES, 'p{p}_uav_chm', 'UAV H (m)',
    ground_truth_col='app_height_m', ground_truth_label='App H (m)'
)
display(stage1_app_table.round(2))

# Quick check: how many valid CHM pixels fall inside a typical 12.5m buffer?
sample_lon, sample_lat = gdf_footprints_clipped.iloc[0][['lon', 'lat']]
with rasterio.open(chm_path) as chm_src:
    to_chm_crs = Transformer.from_crs(WGS84, chm_src.crs, always_xy=True)
    x, y = to_chm_crs.transform(sample_lon, sample_lat)
    buffer_geom = Point(x, y).buffer(BUFFER_RADIUS_M)
    out_image, _ = rio_mask(chm_src, [mapping(buffer_geom)], crop=True, filled=True, nodata=chm_src.nodata)
    n_pixels = np.sum(out_image[0] != chm_src.nodata)
    print(f"~{n_pixels} valid pixels per buffer -> top 1% (P99) is ~{max(1, int(n_pixels*0.01))} pixels, "
          f"P100 is exactly 1 pixel regardless of buffer size")

# Overriding the automatic ranking: P100 is the single max pixel value per
# buffer and is known to be noise-sensitive; P99 is a close second in the
# ranking (MAE=0.98 vs 0.78) and is more stable since it reflects the top 1%
# of pixels rather than one extreme value.
best_uav_percentile = 99
print(f"Using UAV P{best_uav_percentile} (manually selected over the top-ranked "
      f"P100 due to P100's known sensitivity to single-pixel noise)")


CHM extraction complete — total: 6, outside CHM extent: 0, no valid data: 0
Best UAV percentile vs. hypsometer: P98  (MAE=1.09 m, RMSE=1.39 m)


,UAV_Percentile,MAE_m,RMSE_m
0,98,1.093253,1.385770
1,99,1.094547,1.369814
2,100,1.244166,1.762025
3,97,1.411174,1.584581
4,96,2.923999,3.942870
5,95,3.222933,4.253267


Best UAV percentile vs. App Height: P99  (MAE=1.48 m, MedAE=1.17 m, RMSE=1.63 m)


,UAV_Percentile,MAE_m,MedAE_m,RMSE_m
0,99,1.477880,1.167872,1.626554
1,100,1.554165,1.451499,1.822441
2,98,1.594719,1.413500,1.815158
3,97,1.951174,1.999203,2.082056
4,96,3.463999,2.459098,4.516349
5,95,3.762933,2.749848,4.832674


Footprint 1           Footprint 2           Footprint 3            \
            Hyps H (m) UAV H (m)  Hyps H (m) UAV H (m)  Hyps H (m) UAV H (m)   
Percentile                                                                     
100               13.9     17.49        23.7     23.92        15.2     13.57   
99                13.9     15.58        23.7     23.68        15.2     13.25   
98                13.9     14.02        23.7     23.13        15.2     12.86   
97                13.9     13.49        23.7     22.57        15.2     12.74   
96                13.9     13.04        23.7     15.01        15.2     12.57   
95                13.9     12.69        23.7     14.39        15.2     12.48   

           Footprint 4           Footprint 5           Footprint 6            
            Hyps H (m) UAV H (m)  Hyps H (m) UAV H (m)  Hyps H (m) UAV H (m)  
Percentile                                                                    
100               14.1     12.38        17.9     17.66        10.0      9.93  
99                14.1     12.05        17.9     17.32        10.0      9.72  
98                14.1     11.90        17.9     17.20        10.0      9.36  
97                14.1     11.85        17.9     16.94        10.0      8.75  
96                14.1     11.79        17.9     16.65        10.0      8.19  
95                14.1     11.74        17.9     16.50        10.0      7.66

Footprint 1           Footprint 2           Footprint 3            \
             App H (m) UAV H (m)   App H (m) UAV H (m)   App H (m) UAV H (m)   
Percentile                                                                     
100              14.37     17.49       24.81     23.92       15.74     13.57   
99               14.37     15.58       24.81     23.68       15.74     13.25   
98               14.37     14.02       24.81     23.13       15.74     12.86   
97               14.37     13.49       24.81     22.57       15.74     12.74   
96               14.37     13.04       24.81     15.01       15.74     12.57   
95               14.37     12.69       24.81     14.39       15.74     12.48   

           Footprint 4           Footprint 5           Footprint 6            
             App H (m) UAV H (m)   App H (m) UAV H (m)   App H (m) UAV H (m)  
Percentile                                                                    
100              14.39     12.38       18.22     17.66       10.51      9.93  
99               14.39     12.05       18.22     17.32       10.51      9.72  
98               14.39     11.90       18.22     17.20       10.51      9.36  
97               14.39     11.85       18.22     16.94       10.51      8.75  
96               14.39     11.79       18.22     16.65       10.51      8.19  
95               14.39     11.74       18.22     16.50       10.51      7.66

~676 valid pixels per buffer -> top 1% (P99) is ~6 pixels, P100 is exactly 1 pixel regardless of buffer size
Using UAV P99 (manually selected over the top-ranked P100 due to P100's known sensitivity to single-pixel noise)


In [8]:
# ── Stage 2: which GEDI percentile (RH95-RH100) is closest to that UAV percentile? ──
gedi_percentile_candidates = [p for p in range(95, 101) if f'rh{p}' in gdf_footprints_clipped.columns]
missing = [p for p in range(95, 101) if p not in gedi_percentile_candidates]
if missing:
    print(f"Note: GEDI RH{missing} not extracted in notebook 1 — ranking only "
          f"{gedi_percentile_candidates}. To add the rest, add "
          f"\"'rh{{p}}': rh[:, {{p}}]\" for p in range(95, 101) to notebook 1's "
          f"granule-streaming cell, then re-run notebook 1.\n")

uav_best_col = f'p{best_uav_percentile}_uav_chm'

gedi_ranking = pd.DataFrame([
    {
        'GEDI_RH': p,
        'MAE_m': (gdf_footprints_clipped[f'rh{p}'] - gdf_footprints_clipped[uav_best_col]).abs().mean(),
        'RMSE_m': np.sqrt(((gdf_footprints_clipped[f'rh{p}'] - gdf_footprints_clipped[uav_best_col]) ** 2).mean()),
    }
    for p in gedi_percentile_candidates
]).sort_values('MAE_m').reset_index(drop=True)

best_gedi_percentile = int(gedi_ranking.iloc[0]['GEDI_RH'])
print(f"Best GEDI percentile vs. UAV P{best_uav_percentile}: RH{best_gedi_percentile}  "
      f"(MAE={gedi_ranking.iloc[0]['MAE_m']:.2f} m, RMSE={gedi_ranking.iloc[0]['RMSE_m']:.2f} m)")
display(gedi_ranking)

# ── Stage 2 table: percentile x footprint, uav vs. GEDI height ───────
gedi_vs_uav = (
    gdf_footprints_clipped[['reading_order_id'] + [f'p{p}_uav_chm' for p in UAV_PERCENTILES]].merge(
        gdf_footprints_clipped[['reading_order_id'] + [f'rh{p}' for p in gedi_percentile_candidates]],
        on='reading_order_id',
        how='inner'
    )
)

stage2_table = build_percentile_comparison_table(
    gedi_vs_uav, gedi_percentile_candidates, 'rh{p}', 'GEDI H (m)'
)
display(stage2_table.round(2))


Best GEDI percentile vs. UAV P99: RH100  (MAE=4.40 m, RMSE=6.79 m)


,GEDI_RH,MAE_m,RMSE_m
0,100,4.402289,6.787656
1,99,4.803021,7.250588
2,98,5.159688,7.767912
3,97,5.886421,8.276696
4,96,6.586421,8.765228
5,95,7.161421,9.176342


KeyError: 'hypsometer_height_m'

## Correlation Matrix / Other Similarity Comparisons

In [ ]:
# ── Pearson correlation matrix: GEDI RH98, UAV P99, App Height, Hyps. Height ──
def highlight_correlation(val):
    """Cell-background styling: orange/red for negative correlation, green
    for strong (>=0.9) positive correlation, nothing for the diagonal or
    weak correlations."""
    if pd.isna(val):
        return ''
    if abs(val - 1.0) < 1e-9:
        return ''
    if val < 0:
        return 'background-color: #f4b8a3'
    if val >= 0.9:
        return 'background-color: #c6e6c1'
    return ''

corr_input = (
    gdf_footprints_clipped[['reading_order_id', 'p99_uav_chm', 'rh98']]
    .merge(gdf_ground_truth[['reading_order_id', 'app_height_m', 'hypsometer_height_m']],
           on='reading_order_id', how='inner')
    .rename(columns={
        'p99_uav_chm': 'UAV P99 (m)',
        'rh98': 'GEDI RH98 (m)',
        'app_height_m': 'App Height (m)',
        'hypsometer_height_m': 'Hyps. Height (m)',
    })
)

corr_matrix = corr_input[['GEDI RH98 (m)', 'UAV P99 (m)', 'App Height (m)', 'Hyps. Height (m)']].corr()

print(f"Pearson correlation matrix (n={len(corr_input)} footprints)")
styled_corr = corr_matrix.style.format('{:.3f}')
# pandas >= 2.1 renamed Styler.applymap() to Styler.map(); this works either way
try:
    styled_corr = styled_corr.map(highlight_correlation)
except AttributeError:
    styled_corr = styled_corr.applymap(highlight_correlation)
display(styled_corr)

# plot scatter plots for the correlations to get a visual of the similarity between the different height measurements
# closer the line is to y=mx or slope - 1, the 'better' the correlation 
pd.plotting.scatter_matrix(
    corr_input[['GEDI RH98 (m)', 'UAV P99 (m)', 'App Height (m)', 'Hyps. Height (m)']],
    figsize=(8, 8), diagonal='hist'
)
plt.tight_layout()
plt.show()


# ── Cross-check: cosine similarity (mean-centered) ───────────────────────────
# On raw, all-positive height data, cosine similarity is misleadingly high
# almost regardless of actual pattern match. Centering each variable first
# makes this close to Pearson correlation -- treat it as a consistency check
# on the correlation matrix above, not a fundamentally new signal.
height_cols = ['GEDI RH98 (m)', 'UAV P99 (m)', 'App Height (m)', 'Hyps. Height (m)']
X = corr_input[height_cols].values
X_centered = X - X.mean(axis=0)

cosine_matrix = pd.DataFrame(index=height_cols, columns=height_cols, dtype=float)
for i, col_i in enumerate(height_cols):
    for j, col_j in enumerate(height_cols):
        vi, vj = X_centered[:, i], X_centered[:, j]
        cosine_matrix.loc[col_i, col_j] = np.dot(vi, vj) / (np.linalg.norm(vi) * np.linalg.norm(vj))

print("Cosine similarity (mean-centered) -- should closely track the Pearson matrix above")
display(cosine_matrix.astype(float).style.format('{:.3f}'))


# ── Mahalanobis distance: which footprint is the multivariate outlier? ──────
# For each footprint, distance of its [GEDI, UAV, App, Hyps] vector from the
# joint centroid, using all 4 metrics at once -- higher = more atypical.
# CAVEAT: n=6, p=4 makes the raw covariance matrix unstable, so a shrinkage
# estimator (Ledoit-Wolf) is used instead of raw np.cov for numerical
# stability. Treat these as a rough diagnostic, not a rigorous outlier test.
from scipy.spatial.distance import mahalanobis
from sklearn.covariance import LedoitWolf   # pip install scikit-learn if not already available

cov_estimator = LedoitWolf().fit(X)
inv_cov_matrix = np.linalg.inv(cov_estimator.covariance_)
centroid = X.mean(axis=0)

mahal_distances = [mahalanobis(x, centroid, inv_cov_matrix) for x in X]

mahal_table = corr_input[['reading_order_id']].copy()
mahal_table['Mahalanobis_Distance'] = mahal_distances
mahal_table = mahal_table.sort_values('Mahalanobis_Distance', ascending=False).reset_index(drop=True)

print("\nMahalanobis distance of each footprint from the 4-metric centroid")
print("(higher = more atypical jointly across GEDI/UAV/App/Hyps)")
display(mahal_table.round(3))

In [ ]:
%pip install scipy scikit-learn

## Monte Carlo Simulation of New Randomized Footprint Center Coordinate Locations
==========================================================================================

Based on:
  - 'Simulation-Based Correction of Geolocation Errors in GEDI Footprint Positions
     Using Monte Carlo Approach' (Wang et. al 2025)
  - 'The impact of geolocation uncertainty on GEDI tropical forest canopy height
     estimation and change monitoring' (Roy et al. 2021)

Method:
  Each GEDI footprint location is shifted with randomly generated position errors
  modelled using the GEDI geolocation uncertainty (Dubayah et al., 2020a):

      x*_i = x + s_i * cos(theta_i)
      y*_i = y + s_i * sin(theta_i)

  where:
    (x*_i, y*_i) = shifted GEDI footprint center coordinate
    (x, y)       = GEDI product reported footprint center coordinate
    s_i          ~ N(mu=0 m, sigma=10 m)  [geolocation uncertainty]
    theta_i      ~ Uniform(0, 2*pi)       [random direction]
    n            = 300 simulations per footprint

Includes Printout of Map of GEDI Footprints and Simulated Centerpoints and location Distribution of Simulated Center Coordinates

In [ ]:
# ── Simulation parameters ────────────────────────────────────────────────────
N_SIMULATIONS = 300       # number of random shifted positions generated per footprint
SIGMA_M       = 10.0      # GEDI geolocation uncertainty: 1 standard deviation = 10 m
                          # (Dubayah et al. 2020a) — most shots land within 10 m of true position
MU_M          = 0.0       # zero-mean error: no systematic bias assumed, errors are random
SEED          = None        # fixing the seed makes results reproducible run-to-run
                          # remove or change this for true randomness in production
rng           = np.random.default_rng(SEED)

# ── Coordinate reference systems ──────────────────────────────────────────────
# GEDI reports positions in WGS84 (degrees), but we need to apply metre-scale
# offsets. We temporarily project to UTM (metres) to do the geometry, then
# project back to WGS84 for the output.
WGS84      = 'EPSG:4326'
METRIC_CRS = 'EPSG:32610'    # UTM Zone 10N — covers central California
                              # change this if your study area is in a different UTM zone

to_metric = Transformer.from_crs(WGS84, METRIC_CRS, always_xy=True)
to_wgs84  = Transformer.from_crs(METRIC_CRS, WGS84,  always_xy=True)

print(f"Input footprints : {len(gdf_points_clipped):,}")
print(f"Monte Carlo n    : {N_SIMULATIONS}")
print(f"Geolocation σ    : {SIGMA_M} m  (μ = {MU_M} m)")

# ── Core Monte Carlo loop ─────────────────────────────────────────────────────
# For each real GEDI footprint, we simulate N_SIMULATIONS possible "true"
# positions by applying random position errors. This models the uncertainty
# in where the laser actually hit the ground vs where GEDI says it did.
records = []

for fp_idx, row in gdf_points_clipped.iterrows():

    footprint_id = row['shot_number']


    # Step 1: get the reported WGS84 position and convert to metres (UTM)
    lon_orig = row.geometry.x
    lat_orig = row.geometry.y
    x_m, y_m = to_metric.transform(lon_orig, lat_orig)

    # Step 2: sample random radial displacements
    # s_i ~ N(0, 10m) — how far each simulated point is shifted from the original
    # This follows a normal distribution: most shifts are small, few are large
    s_i = rng.normal(loc=MU_M, scale=SIGMA_M, size=N_SIMULATIONS)

    # Step 3: sample random azimuth directions
    # theta_i ~ Uniform(0, 2π) — the direction of each shift is completely random
    # This means errors are equally likely in any compass direction
    theta_i = rng.uniform(low=0.0, high=2 * np.pi, size=N_SIMULATIONS)

    # Step 4: apply the offsets in metric space
    # x*_i = x + s_i * cos(theta_i)   (east-west shift)
    # y*_i = y + s_i * sin(theta_i)   (north-south shift)
    x_star = x_m + s_i * np.cos(theta_i)
    y_star = y_m + s_i * np.sin(theta_i)

    # Step 5: project the shifted positions back to WGS84 (degrees)
    lon_star, lat_star = to_wgs84.transform(x_star, y_star)

    # Step 6: store each simulation as a row in the results list
    for sim_idx in range(N_SIMULATIONS):
        rec = {
            'footprint_id'   : footprint_id,              # which original footprint this belongs to
            'simulation_id'  : sim_idx + 1,           # which simulation iteration (1–300)
            'lon_original'   : lon_orig,              # original reported GEDI position
            'lat_original'   : lat_orig,
            's_i_m'          : s_i[sim_idx],          # radial displacement magnitude in metres
            'theta_i_deg'    : np.degrees(theta_i[sim_idx]),  # shift direction in degrees
            'dx_m'           : s_i[sim_idx] * np.cos(theta_i[sim_idx]),  # east-west component
            'dy_m'           : s_i[sim_idx] * np.sin(theta_i[sim_idx]),  # north-south component
            'lon_shifted'    : lon_star[sim_idx],     # shifted position (the simulated true location)
            'lat_shifted'    : lat_star[sim_idx],
            'geometry'       : Point(lon_star[sim_idx], lat_star[sim_idx]),
        }
        records.append(rec)

# ── Assemble into a single GeoD98
# 
# ataFrame ──────────────────────────────────────
# Result: one row per simulation per footprint
# e.g. 6 footprints × 300 simulations = 1,800 rows
gdf_mc = gpd.GeoDataFrame(records, geometry='geometry', crs=WGS84)

print(f"\nMonte Carlo output shape : {gdf_mc.shape}")
print(f"  ({len(gdf_points_clipped)} footprints × {N_SIMULATIONS} simulations = "
      f"{len(gdf_points_clipped) * N_SIMULATIONS:,} rows)")

# ── Per-footprint summary statistics ─────────────────────────────────────────
# Collapse the 300 simulations per footprint into summary stats:
#   mean shifted position → best estimate of the "true" corrected location
#   std of shifted positions → how uncertain the corrected position is
#   mean radial displacement → average error magnitude across all simulations
summary = (
    gdf_mc
    .groupby('footprint_id')
    .agg(
        lon_original       = ('lon_original',  'first'),
        lat_original       = ('lat_original',  'first'),
        lon_shifted_mean   = ('lon_shifted',   'mean'),   # ensemble mean corrected longitude
        lat_shifted_mean   = ('lat_shifted',   'mean'),   # ensemble mean corrected latitude
        lon_shifted_std    = ('lon_shifted',   'std'),    # spread in longitude across simulations
        lat_shifted_std    = ('lat_shifted',   'std'),    # spread in latitude across simulations
        mean_radial_disp_m = ('s_i_m',         lambda x: np.abs(x).mean()),  # avg displacement
        n_simulations      = ('simulation_id', 'count'),
    )
    .reset_index()
)

# Attach the ensemble-mean corrected position as the geometry
summary_geom = [
    Point(row.lon_shifted_mean, row.lat_shifted_mean)
    for _, row in summary.iterrows()
]
gdf_summary = gpd.GeoDataFrame(summary, geometry=summary_geom, crs=WGS84)

print("\nPer-footprint summary (first 5 rows):")
display(gdf_summary.head())

# Append reading_order_id to gdf_mc and gdf_summary
# ════════════════════════════════════════════════════════════════════════════
if 'reading_order_id' not in gdf_points_clipped.columns:
    raise RuntimeError(
        "reading_order_id missing from gdf_points_clipped — Step A must run "
        "successfully before this step. Re-run from the top of the notebook."
    )

reading_order_lookup = gdf_points_clipped[['shot_number', 'reading_order_id']].drop_duplicates()
reading_order_lookup['shot_number'] = reading_order_lookup['shot_number'].astype(str)

gdf_mc['footprint_id'] = gdf_mc['footprint_id'].astype(str)
gdf_summary['footprint_id'] = gdf_summary['footprint_id'].astype(str)

gdf_mc = gdf_mc.merge(
    reading_order_lookup, left_on='footprint_id', right_on='shot_number', how='left'
).drop(columns='shot_number')

gdf_summary = gdf_summary.merge(
    reading_order_lookup, left_on='footprint_id', right_on='shot_number', how='left'
).drop(columns='shot_number')

n_missing_e = gdf_mc['reading_order_id'].isna().sum()
if n_missing_e > 0:
    print(f"WARNING: {n_missing_e} rows in gdf_mc have no matching reading_order_id")
else:
    print("reading_order_id successfully appended to gdf_mc and gdf_summary")


# ════════════════════════════════════════════════════════════════════════════
# Export results to Excel for inspection/editing
# ════════════════════════════════════════════════════════════════════════════
excel_path1 = os.path.join(data_folder, f'{file_header}_monte_carlo_results.xlsx')

with pd.ExcelWriter(excel_path1, engine='openpyxl') as writer:
    gdf_mc.drop(columns='geometry').to_excel(writer, sheet_name='all_simulations', index=False)
    gdf_summary.drop(columns='geometry').to_excel(writer, sheet_name='per_footprint_summary', index=False)

print(f"Excel file saved to: {excel_path1}")


# ── Diagnostic plots ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left panel: shows the cloud of 300 shifted positions around each original footprint
# Each cluster of blue dots represents the uncertainty envelope for one GEDI shot
ax = axes[0]
ax.scatter(
    gdf_mc['lon_shifted'], gdf_mc['lat_shifted'],
    s=1, alpha=0.15, color='steelblue', label='Shifted positions'
)
ax.scatter(
    gdf_points_clipped.geometry.x, gdf_points_clipped.geometry.y,
    s=40, color='red', zorder=5, label='Original reported positions'
)
# Label each original point with its reading-order number
for _, row in gdf_points_clipped.iterrows():
    ax.annotate(
        str(int(row['reading_order_id'])),
        xy=(row.geometry.x, row.geometry.y),
        xytext=(6, 6),
        textcoords='offset points',
        fontsize=11,
        fontweight='bold',
        color='black',
        zorder=6,
    )
    
ax.set_xlabel('Longitude (°)')
ax.set_ylabel('Latitude (°)')
ax.set_title(f'Monte Carlo Shifted Footprint Positions\n'
             f'(n={N_SIMULATIONS} per footprint, σ={SIGMA_M} m)')
ax.legend(markerscale=3, fontsize=8)

# Right panel: histogram of how large the random shifts were
# Should look like a half-normal distribution centred near 0
# The red line marks the 10 m 1-sigma threshold
ax2 = axes[1]
ax2.hist(np.abs(gdf_mc['s_i_m']), bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax2.axvline(SIGMA_M, color='red', linestyle='--', linewidth=1.5, label=f'σ = {SIGMA_M} m')
ax2.set_xlabel('|Radial displacement| (m)')
ax2.set_ylabel('Frequency')
ax2.set_title('Distribution of Simulated Radial Displacements')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(data_folder, f'{file_header}_monte_carlo.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")

print("\nOutputs ready:")
print("  gdf_mc      – full Monte Carlo simulation GeoDataFrame (all footprints × all iterations)")
print("  gdf_summary – per-footprint ensemble summary with mean-corrected positions")

## Plot Monte Carlo Footprint Circles 
Replicates methods of "The impact of geolocation uncertainty on GEDI tropical forest canopy height estimation and change monitoring" (Roy et al. 2021)  
Prints out simulated buffered footprints in reference to the original footprint location  
Seeds at 42, can set to true random by setting Seed = None  
 - red circle = original reported GEDI 25m footprint
 - black circles = 300 simulated shifted 25m footprints

 Needs editing to be able to generate uav95 for each 300 simulated circles and compare that to the original footprint to find the best match, similar to how it is done below for the 32 iterations.

In [ ]:
# ── Plot shifted footprint circles for each original footprint ────────────────
# Replicates the style of panel (b) in the paper:
#   - red circle = original reported GEDI 25m footprint
#   - black circles = 300 simulated shifted 25m footprints
from shapely.geometry import Point
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

n_footprints = len(gdf_points_clipped)
n_cols = 3
n_rows = math.ceil(n_footprints / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes = axes.flatten()

FOOTPRINT_RADIUS_M = 12.5   # 25m diameter footprint -> 12.5m radius

for plot_idx, (fp_idx, fp_sims) in enumerate(gdf_mc.groupby('footprint_id')):
    ax = axes[plot_idx]

    lon_orig, lat_orig = fp_sims['lon_original'].iloc[0], fp_sims['lat_original'].iloc[0]
    x_orig, y_orig = to_metric.transform(lon_orig, lat_orig)

    for _, sim_row in fp_sims.iterrows():
        x_shift, y_shift = to_metric.transform(sim_row['lon_shifted'], sim_row['lat_shifted'])
        ax.add_patch(Circle((x_shift, y_shift), FOOTPRINT_RADIUS_M, color='black', fill=False, linewidth=0.3, alpha=0.4))

    ax.add_patch(Circle((x_orig, y_orig), FOOTPRINT_RADIUS_M, color='red', fill=False, linewidth=1.5))

    pad = FOOTPRINT_RADIUS_M + SIGMA_M * 3   # show out to ~3 sigma
    ax.set_xlim(x_orig - pad, x_orig + pad)
    ax.set_ylim(y_orig - pad, y_orig + pad)
    ax.set_aspect('equal')
    ax.set_xlabel('UTM Easting (m)', fontsize=8)
    ax.set_ylabel('UTM Northing (m)', fontsize=8)
    ax.set_title(f'Footprint {plot_idx + 1}  (shot {fp_idx})', fontsize=9)
    ax.tick_params(labelsize=7)

plt.suptitle(
    f'Monte Carlo Simulated Footprint Positions\n'
    f'(n={N_SIMULATIONS} per footprint, sigma={SIGMA_M} m, red = original reported position)',
    fontsize=11
)
plt.tight_layout()
plt.savefig(os.path.join(data_folder, f'{file_header}_monte_carlo_circles.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")


# ════════════════════════════════════════════════════════════════════════════
# Extract UAV P98 at each of the 300 simulated positions, then compute error vs GEDI RH98
# ════════════════════════════════════════════════════════════════════════════
BUFFER_RADIUS_M = 12.5
WGS84 = "EPSG:4326"

required_mc_columns = {"footprint_id", "simulation_id", "lon_original", "lat_original", "lon_shifted", "lat_shifted"}
missing_mc_columns = required_mc_columns.difference(gdf_mc.columns)
if missing_mc_columns:
    raise RuntimeError(f"gdf_mc is missing required columns: {sorted(missing_mc_columns)}")
if "rh98" not in gdf_points_clipped.columns:
    raise RuntimeError("The column 'rh98' is missing from gdf_points_clipped.")

print(f"Extracting UAV P98 for {len(gdf_mc):,} simulated positions...")

with rasterio.open(chm_path) as chm_src:
    if chm_src.crs is None:
        raise RuntimeError("The CHM raster does not have a defined CRS.")

    to_chm_crs = Transformer.from_crs(WGS84, chm_src.crs, always_xy=True)
    p98_uav = np.full(len(gdf_mc), np.nan, dtype=float)

    for i, (lon, lat) in enumerate(zip(gdf_mc["lon_shifted"], gdf_mc["lat_shifted"])):
        if not np.isfinite(lon) or not np.isfinite(lat):
            continue

        x_chm, y_chm = to_chm_crs.transform(lon, lat)
        buffer_geom = Point(x_chm, y_chm).buffer(BUFFER_RADIUS_M)

        try:
            # filled=False returns a masked array, which handles nodata safely
            out_image, _ = rio_mask(chm_src, [mapping(buffer_geom)], crop=True, filled=False)
        except ValueError:
            continue   # footprint does not overlap the raster

        vals = out_image[0].compressed()
        vals = vals[np.isfinite(vals)]
        if vals.size > 0:
            p98_uav[i] = np.percentile(vals, 98)

        if (i + 1) % 500 == 0 or (i + 1) == len(gdf_mc):
            print(f"  Processed {i + 1:,} / {len(gdf_mc):,} simulated positions")

gdf_mc["rh98_uav_chm"] = p98_uav
n_valid, n_missing = gdf_mc["rh98_uav_chm"].notna().sum(), gdf_mc["rh98_uav_chm"].isna().sum()
print(f"\nValid UAV P98 extractions : {n_valid:,}")
print(f"Missing UAV P98 values   : {n_missing:,}")


# ── Attach GEDI RH98 and calculate simulation-level errors ──────────────────
gedi_attributes = (
    gdf_points_clipped[["shot_number", "rh98", "reading_order_id"]]
    .drop_duplicates(subset="shot_number")
    .copy()
)
gedi_attributes["shot_number"] = gedi_attributes["shot_number"].astype(str)

rh98_lookup = gedi_attributes.set_index("shot_number")["rh98"]
reading_order_lookup = gedi_attributes.set_index("shot_number")["reading_order_id"]

# map() safely replaces the columns if this cell is rerun
gdf_mc["rh98_gedi"] = gdf_mc["footprint_id"].map(rh98_lookup)
gdf_mc["reading_order_id"] = gdf_mc["footprint_id"].map(reading_order_lookup)

# Signed error: positive = UAV CHM P98 taller than GEDI RH98, negative = shorter
gdf_mc["rh98_error"] = gdf_mc["rh98_uav_chm"] - gdf_mc["rh98_gedi"]
gdf_mc["rh98_abs_error"] = gdf_mc["rh98_error"].abs()

print("\nFirst simulation results:")
display(gdf_mc[[
    "reading_order_id", "footprint_id", "simulation_id", "s_i_m", "theta_i_deg",
    "rh98_gedi", "rh98_uav_chm", "rh98_error", "rh98_abs_error",
]].head(10))


# ── Overall error statistics ─────────────────────────────────────────────────
valid_comparisons = gdf_mc.dropna(subset=["rh98_uav_chm", "rh98_gedi", "rh98_abs_error"]).copy()
if valid_comparisons.empty:
    raise RuntimeError("No valid UAV-GEDI comparisons were produced. Check the CHM extent, CRS, coordinates, and nodata values.")

valid_errors = valid_comparisons["rh98_error"]
print("=" * 65)
print("OVERALL RH98 ERROR STATISTICS - UAV CHM P98 AT SIMULATED LOCATIONS VS GEDI RH98")
print("=" * 65)
print(f"Valid comparisons : {len(valid_errors):,}")
print(f"Mean error (bias) : {valid_errors.mean():.3f} m")
print(f"Error std. dev.   : {valid_errors.std():.3f} m")
print(f"MAE               : {valid_errors.abs().mean():.3f} m")
print(f"RMSE              : {np.sqrt(np.mean(valid_errors**2)):.3f} m")
print(f"Minimum error     : {valid_errors.min():.3f} m")
print(f"Maximum error     : {valid_errors.max():.3f} m")


# ── Best (lowest-absolute-error) simulation per footprint ───────────────────
best_match_idx = valid_comparisons.groupby("footprint_id")["rh98_abs_error"].idxmin()
best_matches = (
    valid_comparisons.loc[best_match_idx].copy()
    .sort_values(["reading_order_id", "footprint_id"], na_position="last")
    .reset_index(drop=True)
)
print(f"\nBest simulated position found for {len(best_matches):,} footprints.")
display(best_matches[[
    "reading_order_id", "footprint_id", "simulation_id",
    "lon_original", "lat_original", "lon_shifted", "lat_shifted",
    "s_i_m", "theta_i_deg", "dx_m", "dy_m",
    "rh98_gedi", "rh98_uav_chm", "rh98_error", "rh98_abs_error",
]])

overall_best = best_matches.loc[best_matches["rh98_abs_error"].idxmin()]
reading_label = int(overall_best["reading_order_id"]) if pd.notna(overall_best["reading_order_id"]) else "unknown"
print(
    "\nBest overall match:\n"
    f"  Reading-order ID : {reading_label}\n"
    f"  Shot number      : {overall_best['footprint_id']}\n"
    f"  Simulation       : {int(overall_best['simulation_id'])}\n"
    f"  Displacement     : {abs(overall_best['s_i_m']):.3f} m\n"
    f"  Bearing          : {overall_best['theta_i_deg']:.2f} deg\n"
    f"  Absolute error   : {overall_best['rh98_abs_error']:.3f} m"
)


# ── Append best-match info to the per-footprint summary ─────────────────────
gdf_summary["footprint_id"] = gdf_summary["footprint_id"].astype(str)

best_columns_to_remove = [
    "best_simulation_id", "best_match_lon", "best_match_lat", "best_displacement_m",
    "best_bearing_deg", "rh98_gedi", "best_match_rh98_uav", "best_match_error", "best_match_abs_error",
]
gdf_summary = gdf_summary.drop(columns=best_columns_to_remove, errors="ignore")

best_match_summary = best_matches[[
    "footprint_id", "simulation_id", "lon_shifted", "lat_shifted",
    "s_i_m", "theta_i_deg", "rh98_gedi", "rh98_uav_chm", "rh98_error", "rh98_abs_error",
]].rename(columns={
    "simulation_id": "best_simulation_id",
    "lon_shifted": "best_match_lon",
    "lat_shifted": "best_match_lat",
    "s_i_m": "best_displacement_m",
    "theta_i_deg": "best_bearing_deg",
    "rh98_uav_chm": "best_match_rh98_uav",
    "rh98_error": "best_match_error",
    "rh98_abs_error": "best_match_abs_error",
})

gdf_summary = gdf_summary.merge(best_match_summary, on="footprint_id", how="left")
print("\nUpdated per-footprint summary:")
display(gdf_summary.head())


# ── Export all results to Excel ──────────────────────────────────────────────
with pd.ExcelWriter(excel_path1, engine="openpyxl", mode="w") as writer:
    gdf_mc.drop(columns="geometry", errors="ignore").to_excel(writer, sheet_name="all_simulations", index=False)
    gdf_summary.drop(columns="geometry", errors="ignore").to_excel(writer, sheet_name="per_footprint_summary", index=False)
    best_matches.drop(columns="geometry", errors="ignore").to_excel(writer, sheet_name="best_matches", index=False)
print(f"\nExcel file updated: {excel_path1}")


# ── Plot original vs. best-matching footprint per GEDI shot ─────────────────
n_footprints = len(best_matches)
if n_footprints == 0:
    raise RuntimeError("There are no valid best matches to plot.")

n_cols = min(3, n_footprints)
n_rows = int(np.ceil(n_footprints / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5 * n_cols, 5.5 * n_rows))
axes = np.atleast_1d(axes).ravel()

with rasterio.open(chm_path) as chm_src:
    to_chm_crs = Transformer.from_crs(WGS84, chm_src.crs, always_xy=True)

    for ax_idx, (_, fp) in enumerate(best_matches.iterrows()):
        ax = axes[ax_idx]
        x_orig, y_orig = to_chm_crs.transform(fp["lon_original"], fp["lat_original"])
        x_best, y_best = to_chm_crs.transform(fp["lon_shifted"], fp["lat_shifted"])

        pad = BUFFER_RADIUS_M * 2.5
        min_x, max_x = min(x_orig, x_best) - pad, max(x_orig, x_best) + pad
        min_y, max_y = min(y_orig, y_best) - pad, max(y_orig, y_best) + pad
        window = window_from_bounds(min_x, min_y, max_x, max_y, transform=chm_src.transform)

        # boundless=True avoids failures near the raster edge
        chm_crop = chm_src.read(1, window=window, boundless=True, masked=True)
        crop_transform = chm_src.window_transform(window)

        ax.imshow(chm_crop, extent=plotting_extent(chm_crop, crop_transform), cmap="YlGn", origin="upper")

        ax.add_patch(Circle((x_orig, y_orig), BUFFER_RADIUS_M, fill=False, edgecolor="red", linewidth=2, label="Original"))
        ax.plot(x_orig, y_orig, "r+", markersize=10, markeredgewidth=2)

        ax.add_patch(Circle((x_best, y_best), BUFFER_RADIUS_M, fill=False, edgecolor="blue", linewidth=2, label="Best simulation"))
        ax.plot(x_best, y_best, "b+", markersize=10, markeredgewidth=2)

        footprint_label = f"#{int(fp['reading_order_id'])}" if pd.notna(fp["reading_order_id"]) else str(fp["footprint_id"])
        ax.set_title(
            f"{footprint_label} | simulation {int(fp['simulation_id'])}\n"
            f"shift={abs(fp['s_i_m']):.2f} m, bearing={fp['theta_i_deg']:.1f} deg\n"
            f"GEDI RH98={fp['rh98_gedi']:.2f} m, UAV P98={fp['rh98_uav_chm']:.2f} m, error={fp['rh98_error']:+.2f} m",
            fontsize=9,
        )
        ax.set_xlabel("Easting")
        ax.set_ylabel("Northing")
        ax.legend(fontsize=7, loc="upper right")
        ax.set_aspect("equal")

for ax_idx in range(n_footprints, len(axes)):
    axes[ax_idx].axis("off")

plt.tight_layout()
best_grid_path = os.path.join(data_folder, f"{file_header}_monte_carlo_best_match_grid.png")
plt.savefig(best_grid_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Best-match grid saved to: {best_grid_path}")

## 32 Equidistant Shifted GEDI Center Coordinates and Best Match Plot 

In [ ]:
# First set some controls to ensure we are in the right coordinate reference system for our calculations
WGS84      = 'EPSG:4326'
METRIC_CRS = 'EPSG:32610'    # UTM Zone 10N - covers central California
                              # change this if your study area is in a different UTM zone

to_metric = Transformer.from_crs(WGS84, METRIC_CRS, always_xy=True)
to_wgs84  = Transformer.from_crs(METRIC_CRS, WGS84,  always_xy=True)

# For each real GEDI footprint, generate a systematic grid of shifted
# positions (4 offsets x 8 bearings + the 1 original position), instead of
# random Monte Carlo shifts. This models the uncertainty in where the laser
# actually hit the ground vs where GEDI says it did.

bearings_degrees = np.array([0, 45, 90, 135, 180, 225, 270, 315])
bearings_radians = np.radians(bearings_degrees)
offsets = np.array([5, 10, 15, 20])

records = []
for fp_idx, row in gdf_points_clipped.iterrows():
    footprint_id = row['shot_number']

    # Original unshifted position in UTM meters
    lon_orig, lat_orig = row.geometry.x, row.geometry.y
    x_m, y_m = to_metric.transform(lon_orig, lat_orig)

    records.append({
        'footprint_id': footprint_id, 'offset': 0, 'bearing_deg': np.nan,
        'dx_m': 0.0, 'dy_m': 0.0,
        'lon_original': lon_orig, 'lat_original': lat_orig,
        'lon_shifted': lon_orig, 'lat_shifted': lat_orig,
        'geometry': Point(lon_orig, lat_orig),
    })

    for d in offsets:
        for theta_deg, theta_rad in zip(bearings_degrees, bearings_radians):
            dx, dy = d * np.cos(theta_rad), d * np.sin(theta_rad)
            x_shifted, y_shifted = x_m + dx, y_m + dy
            lon_shifted, lat_shifted = to_wgs84.transform(x_shifted, y_shifted)

            records.append({
                'footprint_id': footprint_id, 'offset': d, 'bearing_deg': theta_deg,
                'dx_m': dx, 'dy_m': dy,
                'lon_original': lon_orig, 'lat_original': lat_orig,
                'lon_shifted': lon_shifted, 'lat_shifted': lat_shifted,
                'geometry': Point(lon_shifted, lat_shifted),
            })

gdf_shift = gpd.GeoDataFrame(records, geometry='geometry', crs=WGS84)

print(f"Generated {len(gdf_shift):,} rows "
      f"({gdf_shift['footprint_id'].nunique():,} footprints x "
      f"{len(offsets) * len(bearings_degrees) + 1} positions each: 1 original + 32 shifted)")

# ── Extract UAV P98 at each of the 33 positions, then compute epsilon_d_theta ──
BUFFER_RADIUS_M = 12.5   # 25 m GEDI footprint diameter -> 12.5 m radius buffer

with rasterio.open(chm_path) as chm_src:
    chm_nodata = chm_src.nodata
    to_chm_crs = Transformer.from_crs(WGS84, chm_src.crs, always_xy=True)

    p98_uav = np.full(len(gdf_shift), np.nan)

    for i, (lon, lat) in enumerate(zip(gdf_shift['lon_shifted'], gdf_shift['lat_shifted'])):
        x_chm, y_chm = to_chm_crs.transform(lon, lat)                     # into the CHM's CRS
        buffer_geom = Point(x_chm, y_chm).buffer(BUFFER_RADIUS_M)          # 12.5 m buffer around the point

        try:
            out_image, _ = rio_mask(chm_src, [mapping(buffer_geom)], crop=True, filled=True, nodata=chm_nodata)
        except ValueError:
            continue

        vals = out_image[0]
        vals = vals[vals != chm_nodata] if chm_nodata is not None else vals.flatten()
        vals = vals[~np.isnan(vals)]

        if vals.size:
            p98_uav[i] = np.percentile(vals, 98)   # P98 = value below which 98% of CHM heights in the buffer fall

        if (i + 1) % 500 == 0:
            print(f"  Processed {i + 1:,} / {len(gdf_shift):,} positions...")

# NOTE: these three lines belong AFTER the loop, not inside it — previously
# they were over-indented, so the column assignment, GEDI lookup, and error
# calc were all being redone (and the head(10) table reprinted) on every
# single one of the 500+ iterations instead of once at the end.
gdf_shift['p98_uav_chm'] = p98_uav

gedi_rh98_lookup = gdf_points_clipped.set_index('shot_number')['rh98']
gdf_shift['rh98_gedi'] = gdf_shift['footprint_id'].map(gedi_rh98_lookup)
gdf_shift['epsilon_d_theta'] = gdf_shift['p98_uav_chm'] - gdf_shift['rh98_gedi']

print(gdf_shift[['footprint_id', 'offset', 'bearing_deg', 'p98_uav_chm', 'rh98_gedi', 'epsilon_d_theta']].head(10))

with pd.ExcelWriter(excel_path1, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    gdf_shift.drop(columns='geometry', errors='ignore').to_excel(writer, sheet_name='shift_experiment_all', index=False)

print(f"\nExcel file updated with shift-experiment results: {excel_path1}")

# ════════════════════════════════════════════════════════════════════════════
# Find the best-matching shifted coordinate per footprint
# ════════════════════════════════════════════════════════════════════════════
# For each footprint, identify which of the 33 positions (1 original + 32
# shifted) produced a UAV P98 value closest to the GEDI RH98 - i.e. the
# offset distance/bearing that minimizes |epsilon_d_theta|.

gdf_shift['abs_epsilon'] = gdf_shift['epsilon_d_theta'].abs()

best_matches = (
    gdf_shift
    .loc[gdf_shift.groupby('footprint_id')['abs_epsilon'].idxmin()]
    .reset_index(drop=True)
)

print(f"Best-matching shifted coordinate found for {len(best_matches):,} footprints.\n")

for _, fp in best_matches.iterrows():
    print(
        f"Footprint {fp['footprint_id']}: "
        f"best offset = {fp['offset']:.0f} m at bearing {fp['bearing_deg']:.0f} deg  |  "
        f"shifted coord = ({fp['lon_shifted']:.6f}, {fp['lat_shifted']:.6f})  |  "
        f"GEDI RH98 = {fp['rh98_gedi']:.2f} m, UAV P98 = {fp['p98_uav_chm']:.2f} m, "
        f"error = {fp['epsilon_d_theta']:+.2f} m"
    )

# ════════════════════════════════════════════════════════════════════════════
# Grid plot: original vs. best-matching shifted buffer, per footprint
# ════════════════════════════════════════════════════════════════════════════
from rasterio.windows import from_bounds as window_from_bounds
from matplotlib.patches import Circle
import matplotlib.pyplot as plt

n_footprints = len(best_matches)
n_cols = min(3, n_footprints)
n_rows = int(np.ceil(n_footprints / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5 * n_cols, 5.5 * n_rows))
axes = np.atleast_1d(axes).flatten()

with rasterio.open(chm_path) as chm_src:
    to_chm_crs = Transformer.from_crs(WGS84, chm_src.crs, always_xy=True)

    for ax_idx, (_, fp) in enumerate(best_matches.iterrows()):
        ax = axes[ax_idx]

        x_orig, y_orig = to_chm_crs.transform(fp['lon_original'], fp['lat_original'])
        x_best, y_best = to_chm_crs.transform(fp['lon_shifted'], fp['lat_shifted'])

        pad = BUFFER_RADIUS_M * 2.5
        min_x, max_x = min(x_orig, x_best) - pad, max(x_orig, x_best) + pad
        min_y, max_y = min(y_orig, y_best) - pad, max(y_orig, y_best) + pad

        window = window_from_bounds(min_x, min_y, max_x, max_y, transform=chm_src.transform)
        chm_crop = chm_src.read(1, window=window)
        crop_transform = chm_src.window_transform(window)

        extent = (
            crop_transform.c, crop_transform.c + chm_crop.shape[1] * crop_transform.a,
            crop_transform.f + chm_crop.shape[0] * crop_transform.e, crop_transform.f,
        )

        ax.imshow(chm_crop, extent=extent, cmap='YlGn', origin='upper')

        ax.add_patch(Circle((x_orig, y_orig), BUFFER_RADIUS_M, fill=False, edgecolor='red', linewidth=2, label='Original'))
        ax.plot(x_orig, y_orig, 'r+', markersize=10, markeredgewidth=2)

        ax.add_patch(Circle((x_best, y_best), BUFFER_RADIUS_M, fill=False, edgecolor='blue', linewidth=2, label='Best match'))
        ax.plot(x_best, y_best, 'b+', markersize=10, markeredgewidth=2)

        bearing_str = f"{fp['bearing_deg']:.0f} deg" if pd.notna(fp['bearing_deg']) else "n/a"

        ax.set_title(
            f"Footprint {fp['footprint_id']}  |  offset={fp['offset']:.0f}m  bearing={bearing_str}\n"
            f"GEDI RH98={fp['rh98_gedi']:.2f}m  UAV P98={fp['p98_uav_chm']:.2f}m  "
            f"error={fp['epsilon_d_theta']:+.2f}m",
            fontsize=10
        )
        ax.set_xlabel('Easting (m)')
        ax.set_ylabel('Northing (m)')
        ax.legend(fontsize=7, loc='upper right')
        ax.set_aspect('equal')

for ax_idx in range(n_footprints, len(axes)):
    axes[ax_idx].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(data_folder, f'{file_header}_best_match_grid.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Best-match grid figure saved.")